# Inline NIR Diversion-Control Performance Verification

## Purpose

This notebook provides a first workflow for inline NIR diversion-control performance verification using synthetic example data. It compares a prospective new inline run against historical old-method performance benchmarks, extracted reference samples, precomputed process_repeatability, diversion decision zones, and spectral diagnostics.

## Why paired USP <1010> comparison is not feasible for inline diversion NIR

A classical paired old-vs-new USP <1010> procedure comparison is not feasible for this inline diversion-control setting because the old and changed inline methods cannot measure the exact same dynamic material stream under identical process conditions. The verification pathway therefore relies on historical old-method benchmarks and a prospective new-method process run with reference-aligned samples and lifecycle diagnostics.

## Comparative-study pathway

The workflow is organized around:

- historical old-method performance benchmark
- prospective new-method process run
- extracted reference samples
- accuracy versus reference
- precomputed process_repeatability
- diversion decision-risk assessment
- spectral diagnostics and lifecycle monitoring

## Input configuration

In [11]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from nir_cp.inline_diversion import (
    calculate_reference_errors,
    classify_diversion_zone,
    compare_new_to_historical_run_summary,
    inline_verification_decision,
    summarize_accuracy_vs_reference,
    summarize_diversion_decisions,
    summarize_spectral_diagnostics,
)
from nir_cp.notebook_export import export_notebook_pdf
from nir_cp.process_repeatability import calculate_process_repeatability, process_repeatability_summary_frame
from nir_cp.report_equations import display_equation, display_equation_set
from nir_cp.report_tables import display_report_dataframe
from nir_cp.reporting_text import inline_verification_summary_text

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "examples").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

HISTORICAL_PATH = PROJECT_ROOT / "examples" / "inline_historical_runs_example.csv"
NEW_RUN_PATH = PROJECT_ROOT / "examples" / "inline_new_run_reference_samples_example.csv"
RUN_SUMMARY_PATH = PROJECT_ROOT / "examples" / "inline_run_summary_example.csv"

# Preliminary placeholders. Replace or confirm with CP-approved method-specific criteria before GMP use.
D_INLINE = 0.50
K_INLINE = 1.50
GUARD_BAND = 0.50
Q_RESIDUAL_LIMIT = 1.20
HOTELLING_T2_LIMIT = 8.00
PROCESS_REPEATABILITY_WINDOW_POINTS = 31
PROCESS_REPEATABILITY_MIN_POINTS = 11
PROCESS_REPEATABILITY_WINDOW_SECONDS = None
EXCLUDE_INVALID_SPECTRA = True
EXCLUDE_DIAGNOSTIC_EXCEEDANCES = False
EXPORT_REPORT = True

METHOD_NAME = "Inline NIR diversion-control method"

# Statistical theory and decision rules

<!-- SVG equation metadata for tests: $$$$ Y_{NIR,i} RMSEP \beta_0 -->

## Study design

Classical paired USP <1010> procedure comparison is not technically feasible for this inline method because the old and new inline methods cannot measure the same dynamic process stream under identical conditions. The inline comparative pathway therefore uses historical current-method performance, prospective new-method data, extracted reference samples, spectral diagnostics, diversion-zone assessment, and process_repeatability.

The notebook calls tested functions under `src/nir_cp/` and does not implement independent pass/fail logic.

## Accuracy versus reference

For extracted reference samples, the prediction error is:

In [ ]:
_ = display_equation(r"e_i = Y_{NIR,i} - Y_{REF,i}")

The reported agreement metrics include:

In [ ]:
_ = display_equation_set([
    r"bias = \frac{1}{n}\sum_{i=1}^{n}e_i",
    r"MAE = \frac{1}{n}\sum_{i=1}^{n}|e_i|",
    r"RMSEP = \sqrt{\frac{1}{n}\sum_{i=1}^{n}e_i^2}",
])

These metrics assess agreement between inline NIR predictions and extracted reference-sample results. They do not by themselves constitute a paired old-versus-new USP <1010> procedure comparison.

## Diversion-zone assessment

NIR predictions are classified relative to the lower and upper diversion limits:

In [ ]:
_ = display_equation_set([
    r"Y_{NIR,i} < L_{div}",
    r"L_{div} \le Y_{NIR,i} \le U_{div}",
    r"Y_{NIR,i} > U_{div}",
])

A guard band, when used, flags predictions that remain within the diversion limits but are close enough to a limit to need additional review.

## process_repeatability by local linear detrending

The local model is:

In [ ]:
_ = display_equation(r"Y(t) = \beta_0 + \beta_1t + \epsilon")

The residual is:

In [ ]:
_ = display_equation(r"r_i = Y_i - \hat{Y}_i")

The process_repeatability estimate is:

In [ ]:
_ = display_equation(r"s_r = \sqrt{\frac{\sum_{i=1}^{n_r}(r_i-\bar{r})^2}{n_r-1}}")

This metric estimates short-term residual variability after removing slow local process trends. It is not the raw full-run SD. It is not the SD of NIR-reference errors. Exclusions must be predefined, for example invalid spectra or predefined diagnostic exceedances.

## Values used in this notebook

The table below is generated from the notebook variables. Calculations and final pass/fail decisions are performed by tested functions under `src/nir_cp/`.

In [12]:
values_used = {
    "method name": METHOD_NAME,
    "D_INLINE": D_INLINE,
    "K_INLINE": K_INLINE,
    "guard band": GUARD_BAND,
    "process_repeatability mode": "local_linear_detrending",
    "window_points": PROCESS_REPEATABILITY_WINDOW_POINTS,
    "window_seconds": PROCESS_REPEATABILITY_WINDOW_SECONDS,
    "min_points": PROCESS_REPEATABILITY_MIN_POINTS,
    "exclude_invalid_spectra": EXCLUDE_INVALID_SPECTRA,
    "exclude_diagnostic_exceedances": EXCLUDE_DIAGNOSTIC_EXCEEDANCES,
    "q_residual_limit": Q_RESIDUAL_LIMIT,
    "hotelling_t2_limit": HOTELLING_T2_LIMIT,
}

display_report_dataframe(pd.DataFrame([values_used]), transpose_if_one_row=True);

parameter,value
method name,Inline NIR diversion-control method
D_INLINE,0.500
K_INLINE,1.500
guard band,0.500
process_repeatability mode,local_linear_detrending
window_points,31
window_seconds,None
min_points,11
exclude_invalid_spectra,True
exclude_diagnostic_exceedances,False


## Load historical and new-run data

In [13]:
historical_df = pd.read_csv(HISTORICAL_PATH)
new_run_df = pd.read_csv(NEW_RUN_PATH)
run_summary_df = pd.read_csv(RUN_SUMMARY_PATH)

display_report_dataframe(historical_df, max_rows=10);
display_report_dataframe(new_run_df, max_rows=10);
display_report_dataframe(run_summary_df);

run_id,batch_id,timestamp,nir_prediction,reference_result,reference_sample_id,diversion_lower_limit,diversion_upper_limit,q_residual,hotelling_t2,valid_spectrum
OLD-INL-001,SYN-IB001,2026-04-01T08:00:00,99.600,99.700,REF-H001,97.000,103.000,0.420,4.800,True
OLD-INL-001,SYN-IB001,2026-04-01T08:15:00,100.100,100.000,REF-H002,97.000,103.000,0.550,5.200,True
OLD-INL-001,SYN-IB001,2026-04-01T08:30:00,100.500,100.600,REF-H003,97.000,103.000,0.610,5.900,True
OLD-INL-001,SYN-IB001,2026-04-01T08:45:00,99.200,99.300,REF-H004,97.000,103.000,0.740,6.100,True
OLD-INL-001,SYN-IB001,2026-04-01T09:00:00,101.000,100.800,REF-H005,97.000,103.000,0.680,6.400,True
OLD-INL-001,SYN-IB001,2026-04-01T09:15:00,98.900,99.000,REF-H006,97.000,103.000,0.830,6.800,True
OLD-INL-001,SYN-IB001,2026-04-01T09:30:00,100.400,100.500,REF-H007,97.000,103.000,0.710,5.700,True
OLD-INL-001,SYN-IB001,2026-04-01T09:45:00,99.800,99.700,REF-H008,97.000,103.000,0.660,5.500,True
OLD-INL-001,SYN-IB001,2026-04-01T10:00:00,100.700,100.600,REF-H009,97.000,103.000,0.790,6.000,True
OLD-INL-001,SYN-IB001,2026-04-01T10:15:00,99.400,99.500,REF-H010,97.000,103.000,0.870,6.600,True


run_id,batch_id,timestamp,nir_prediction,reference_result,reference_sample_id,diversion_lower_limit,diversion_upper_limit,q_residual,hotelling_t2,valid_spectrum
NEW-INL-001,SYN-NB001,2026-04-10T08:00:00,99.700,99.600,REF-N001,97.000,103.000,0.500,5.100,True
NEW-INL-001,SYN-NB001,2026-04-10T08:12:00,100.200,100.100,REF-N002,97.000,103.000,0.620,5.800,True
NEW-INL-001,SYN-NB001,2026-04-10T08:24:00,100.600,100.500,REF-N003,97.000,103.000,0.740,6.300,True
NEW-INL-001,SYN-NB001,2026-04-10T08:36:00,99.100,99.200,REF-N004,97.000,103.000,0.860,6.900,True
NEW-INL-001,SYN-NB001,2026-04-10T08:48:00,101.000,100.900,REF-N005,97.000,103.000,0.820,6.700,True
NEW-INL-001,SYN-NB001,2026-04-10T09:00:00,98.900,99.000,REF-N006,97.000,103.000,0.960,7.500,True
NEW-INL-001,SYN-NB001,2026-04-10T09:12:00,100.300,100.200,REF-N007,97.000,103.000,0.710,6.100,True
NEW-INL-001,SYN-NB001,2026-04-10T09:24:00,99.600,99.700,REF-N008,97.000,103.000,0.780,6.500,True
NEW-INL-001,SYN-NB001,2026-04-10T09:36:00,100.800,100.700,REF-N009,97.000,103.000,0.800,6.400,True
NEW-INL-001,SYN-NB001,2026-04-10T09:48:00,99.400,99.500,REF-N010,97.000,103.000,1.050,8.200,True


method_status,run_id,batch_id,n_reference_samples,process_repeatability,mean_bias,rmsep,mae,invalid_spectrum_rate,q_residual_exceedance_rate,hotelling_t2_exceedance_rate
old,OLD-INL-001,SYN-IB001,10,0.360,-0.010,0.120,0.100,0.000,0.000,0.000
old,OLD-INL-002,SYN-IB002,10,0.420,-0.020,0.130,0.110,0.000,0.000,0.000
old,OLD-INL-003,SYN-IB003,10,0.390,-0.010,0.110,0.100,0.000,0.000,0.000
new,NEW-INL-001,SYN-NB001,16,0.440,0.010,0.130,0.110,0.060,0.060,0.060


## Accuracy versus reference

In [14]:
historical_errors = calculate_reference_errors(historical_df)
new_run_errors = calculate_reference_errors(new_run_df)

historical_accuracy = summarize_accuracy_vs_reference(historical_df)
new_accuracy = summarize_accuracy_vs_reference(new_run_df)

display_report_dataframe(pd.DataFrame([historical_accuracy], index=["historical_old_method"]), index=True, transpose_if_one_row=True);
display_report_dataframe(pd.DataFrame([new_accuracy], index=["new_method_run"]), index=True, transpose_if_one_row=True);
display_report_dataframe(new_run_errors[["reference_sample_id", "nir_prediction", "reference_result", "error", "absolute_error"]], max_rows=10);

parameter,value
n,30.000
mean_bias,0.000
sd_error,0.111
mae,0.107
rmsep,0.110
min_error,-0.100
max_error,0.200


parameter,value
n,16.000
mean_bias,0.013
sd_error,0.120
mae,0.112
rmsep,0.117
min_error,-0.200
max_error,0.200


reference_sample_id,nir_prediction,reference_result,error,absolute_error
REF-N001,99.700,99.600,0.100,0.100
REF-N002,100.200,100.100,0.100,0.100
REF-N003,100.600,100.500,0.100,0.100
REF-N004,99.100,99.200,-0.100,0.100
REF-N005,101.000,100.900,0.100,0.100
REF-N006,98.900,99.000,-0.100,0.100
REF-N007,100.300,100.200,0.100,0.100
REF-N008,99.600,99.700,-0.100,0.100
REF-N009,100.800,100.700,0.100,0.100
REF-N010,99.400,99.500,-0.100,0.100


## Diversion-zone assessment

In [15]:
new_zones = classify_diversion_zone(new_run_df, guard_band=GUARD_BAND)
historical_zones = classify_diversion_zone(historical_df, guard_band=GUARD_BAND)

display_report_dataframe(summarize_diversion_decisions(historical_zones));
display_report_dataframe(summarize_diversion_decisions(new_zones));
display_report_dataframe(new_zones[["reference_sample_id", "nir_prediction", "diversion_lower_limit", "diversion_upper_limit", "diversion_zone"]], max_rows=10);

diversion_zone,count,proportion
within_limits,30,1.000


diversion_zone,count,proportion
within_limits,16,1.000


reference_sample_id,nir_prediction,diversion_lower_limit,diversion_upper_limit,diversion_zone
REF-N001,99.700,97.000,103.000,within_limits
REF-N002,100.200,97.000,103.000,within_limits
REF-N003,100.600,97.000,103.000,within_limits
REF-N004,99.100,97.000,103.000,within_limits
REF-N005,101.000,97.000,103.000,within_limits
REF-N006,98.900,97.000,103.000,within_limits
REF-N007,100.300,97.000,103.000,within_limits
REF-N008,99.600,97.000,103.000,within_limits
REF-N009,100.800,97.000,103.000,within_limits
REF-N010,99.400,97.000,103.000,within_limits


## Calculated process_repeatability by local linear detrending

For the prospective new run, process_repeatability is calculated as the sample standard deviation of local-linear-detrended NIR prediction residuals. The configured exclusions are predefined validity and diagnostic rules only; statistical outlier removal is not applied.

In [16]:
calculated_repeatability = calculate_process_repeatability(
    new_run_df,
    window_seconds=PROCESS_REPEATABILITY_WINDOW_SECONDS,
    window_points=PROCESS_REPEATABILITY_WINDOW_POINTS,
    min_points=PROCESS_REPEATABILITY_MIN_POINTS,
    exclude_invalid=EXCLUDE_INVALID_SPECTRA,
    q_col="q_residual",
    q_limit=Q_RESIDUAL_LIMIT,
    t2_col="hotelling_t2",
    t2_limit=HOTELLING_T2_LIMIT,
    exclude_diagnostics=EXCLUDE_DIAGNOSTIC_EXCEEDANCES,
)

display_report_dataframe(process_repeatability_summary_frame(calculated_repeatability));
display_report_dataframe(calculated_repeatability["residuals"], max_rows=10);

process_repeatability,mode,unit,window_seconds,window_points,min_points,exclude_invalid,exclude_diagnostics,n_input_rows,n_rows_used_for_centres,n_residuals,n_skipped,percent_rows_used,mean_residual,residual_median,residual_mad
0.804,local_linear_detrending,assay_units,None,31,11,True,False,16,15,15,0,93.750,-0.000,0.269,0.519


original_index,timestamp,elapsed_seconds,observed,fitted,residual,local_slope,local_intercept,n_points_window,window_start_time,window_end_time
0,2026-04-10 08:00:00,0.000,99.700,99.913,-0.213,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00
1,2026-04-10 08:12:00,720.000,100.200,99.931,0.269,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00
2,2026-04-10 08:24:00,1440.000,100.600,99.949,0.651,0.000,99.912,15,2026-04-10 08:00:00,2026-04-10 10:48:00
3,2026-04-10 08:36:00,2160.000,99.100,99.967,-0.867,0.000,99.912,15,2026-04-10 08:00:00,2026-04-10 10:48:00
4,2026-04-10 08:48:00,2880.000,101.000,99.985,1.015,0.000,99.912,15,2026-04-10 08:00:00,2026-04-10 10:48:00
5,2026-04-10 09:00:00,3600.000,98.900,100.004,-1.104,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00
6,2026-04-10 09:12:00,4320.000,100.300,100.022,0.278,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00
7,2026-04-10 09:24:00,5040.000,99.600,100.040,-0.440,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00
8,2026-04-10 09:36:00,5760.000,100.800,100.058,0.742,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00
9,2026-04-10 09:48:00,6480.000,99.400,100.076,-0.676,0.000,99.913,15,2026-04-10 08:00:00,2026-04-10 10:48:00


## Precomputed process_repeatability comparison

The historical comparison below uses precomputed process_repeatability values from the run summary file. Historical comparison can use precomputed historical values or calculated values if the same approved algorithm is applied consistently across historical and new runs. This notebook does not fail if historical raw time series are not sufficient for calculated old-run values.

In [17]:
repeatability_comparison = compare_new_to_historical_run_summary(run_summary_df)
display_report_dataframe(pd.DataFrame([repeatability_comparison]), transpose_if_one_row=True);

parameter,value
historical_n_runs,3.000
new_n_runs,1.000
historical_mean_process_repeatability,0.390
historical_max_process_repeatability,0.420
new_process_repeatability,0.440
ratio_new_to_historical_mean,1.128
ratio_new_to_historical_max,1.048


## Spectral diagnostics

In [18]:
historical_diagnostics = summarize_spectral_diagnostics(
    historical_df,
    q_limit=Q_RESIDUAL_LIMIT,
    t2_limit=HOTELLING_T2_LIMIT,
)
new_diagnostics = summarize_spectral_diagnostics(
    new_run_df,
    q_limit=Q_RESIDUAL_LIMIT,
    t2_limit=HOTELLING_T2_LIMIT,
)

display_report_dataframe(pd.DataFrame([historical_diagnostics], index=["historical_old_method"]), index=True, transpose_if_one_row=True);
display_report_dataframe(pd.DataFrame([new_diagnostics], index=["new_method_run"]), index=True, transpose_if_one_row=True);

parameter,value
n,30.000
invalid_spectrum_rate,0.000
q_residual_exceedance_rate,0.000
hotelling_t2_exceedance_rate,0.000


parameter,value
n,16.000
invalid_spectrum_rate,0.062
q_residual_exceedance_rate,0.062
hotelling_t2_exceedance_rate,0.188


## Overall preliminary decision

In [19]:
decision = inline_verification_decision(
    accuracy_summary=new_accuracy,
    repeatability_comparison=repeatability_comparison,
    d_inline=D_INLINE,
    k_inline=K_INLINE,
)
decision.update(
    {
        "process_repeatability_mode": calculated_repeatability["mode"],
        "n_residuals": calculated_repeatability["summary"]["n_residuals"],
        "process_repeatability_window_points": calculated_repeatability["parameters"]["window_points"],
        "process_repeatability_window_seconds": calculated_repeatability["parameters"]["window_seconds"],
    }
)

display(Markdown(inline_verification_summary_text(decision, method_name=METHOD_NAME)))
display_report_dataframe(pd.DataFrame([decision]), transpose_if_one_row=True);

Inline NIR diversion-control method met the predefined criteria for preliminary inline diversion-control verification. The observed mean bias versus reference was 0.013, compared with the preliminary limit +/-0.500. new-run process_repeatability was calculated from local linear detrending residuals (window_points=31, n_residuals=15). The observed ratio to the historical mean was 1.128, compared with the preliminary limit k=1.500. This statement uses the reported process_repeatability basis and preliminary criteria that must be confirmed or replaced by CP-approved criteria before GMP use.

parameter,value
accuracy_pass,True
repeatability_pass,True
overall_pass,True
decision_text,Preliminary inline diversion-control verification met the predefined accuracy and process repeatability criteria. These criteria must be confirmed or replaced by CP-approved criteria before GMP use.
mean_bias,0.013
d_inline,0.500
ratio_new_to_historical_mean,1.128
k_inline,1.500
process_repeatability_mode,local_linear_detrending
n_residuals,15


## Export report

Set `EXPORT_REPORT = True` in the configuration cell to export this notebook to PDF. Export is disabled by default.

In [20]:
if EXPORT_REPORT:
    notebook_path = PROJECT_ROOT / "notebooks" / "04_inline_diversion_performance_verification.ipynb"
    pdf_path = PROJECT_ROOT / "reports" / "pdf" / "04_inline_diversion_performance_verification.pdf"
    exported_pdf = export_notebook_pdf(notebook_path, pdf_path, hide_code=True, keep_html=True)
    display(Markdown(f"Exported PDF report: `{exported_pdf}`"))
else:
    display(Markdown("PDF export skipped because `EXPORT_REPORT` is `False`."))

Exported PDF report: `c:\Users\Erik\Documents\Projects\nir-cp-comparability\reports\pdf\04_inline_diversion_performance_verification.pdf`